In [1]:
from pymol import cmd
cmd.feedback("disable", "all", "everything")  # Disables all output, comment this if something doesn't go as expected
import pandas as pd
import os

In [2]:
# ALL_RESULTS_TRIPLETS = pd.read_csv("../processed/protein_priorization/triplets.tsv", sep='\t')
# ALL_RESULTS_TRIPLETS[(ALL_RESULTS_TRIPLETS['RMSD_A'] < 2) & (ALL_RESULTS_TRIPLETS['RMSD_B'] < 2) & (ALL_RESULTS_TRIPLETS['RMSD_C'] < 2)]

In [14]:
ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_prioritization/pairs.tsv", sep='\t')
ALL_RESULTS_PAIRS[(ALL_RESULTS_PAIRS['Protein RMSD'] > 10) & (ALL_RESULTS_PAIRS['PocketVec distance'] < 0.17)]

,Protein1,Pocket1,InterPro-Pocket1,Protein2,Pocket2,InterPro-Pocket2,PocketVec distance,Protein RMSD,Protein SEQ ID (NW),Protein SEQ ID (CO),Dist PDB 1,Dist PDB 2,Dist Alphafill 1,Dist AlphaFill 2,P2Rank score 1,P2Rank score 2
1,P9WFT1,alphafold2_P9WFT1_model_0_pocket_2,Catalytic Domain (ATP Binding Site),P9WFU5,alphafold3_P9WFU5_model_2_pocket_1,Catalytic Domain (ATP Binding Site);Other too ...,0.117,18.21,29.0055,12.29,NaN,NaN,10.547,0.762,5.96,17.36
4,P9WFV7,swissmodel_P9WFV7_model_1_pocket_2,Anticodon Binding Domain,P9WFW5,chai1_P9WFW5_model_4_pocket_2,Catalytic Domain (ATP Binding Site);Other too ...,0.121,17.16,25.0000,12.75,NaN,NaN,24.070,12.217,3.70,4.69
7,P9WFV3,alphafold3_P9WFV3_model_4_pocket_3,Catalytic Domain (ATP Binding Site),P9WFW1,swissmodel_P9WFW1_model_0_pocket_3,Catalytic Domain (ATP Binding Site),0.123,10.85,34.6320,13.66,NaN,NaN,15.873,4.442,5.86,5.31
11,P9WFT1,alphafold3_P9WFT1_model_0_pocket_2,Catalytic Domain (ATP Binding Site),P9WN61,chai1_P9WN61_model_4_pocket_2,Other too broad/unspecified functional entities,0.127,13.74,26.1421,12.83,NaN,NaN,10.599,9.526,5.55,3.80
15,P9WFS9,alphafold3_P9WFS9_model_4_pocket_3,Catalytic Domain (ATP Binding Site),P9WFU1,chai1_P9WFU1_model_0_pocket_2,Catalytic Domain (ATP Binding Site),0.127,22.52,27.7092,13.90,NaN,NaN,10.105,12.423,7.70,9.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1474,P9WFT5,alphafold2_P9WFT5_model_0_pocket_3,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFV9,chai1_P9WFV9_model_2_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,0.169,15.28,29.2576,11.68,NaN,NaN,20.571,21.407,6.92,3.92
1475,P9WFS9,alphafold3_P9WFS9_model_3_pocket_4,Catalytic Domain (ATP Binding Site),P9WFT1,alphafold3_P9WFT1_model_0_pocket_2,Catalytic Domain (ATP Binding Site),0.169,12.35,34.1463,9.76,NaN,NaN,9.997,10.599,7.82,5.55
1476,P9WFS9,alphafold3_P9WFS9_model_4_pocket_2,Catalytic Domain (ATP Binding Site);Editing Do...,P9WFT9,swissmodel_P9WFT9_model_0_pocket_2,Anticodon Binding Domain,0.169,11.13,27.3852,13.78,NaN,NaN,1.588,14.631,10.46,3.06
1479,P9WFT1,alphafold3_P9WFT1_model_4_pocket_2,Catalytic Domain (ATP Binding Site),P9WFW7,swissmodel_P9WFW7_model_0_pocket_1,NaN,0.169,18.89,31.6547,17.65,NaN,NaN,10.806,1.716,6.76,24.28


In [7]:
def prepare_pymol_session(pocket1, pocket2, outdir):

    st1 = "_".join(pocket1.split("_")[:4])
    p1 = "_".join(pocket1.split("_")[4:])
    st2 = "_".join(pocket2.split("_")[:4])
    p2 = "_".join(pocket2.split("_")[4:])

    PATH_TO_STRUCTURES = "../processed/aligned_relaxed_structures/"
    PATH_TO_POCKETS = "../processed/detected_pockets/"


    # Define some colors
    COLORS = ['wheat', 'grey', 'skyblue']

    # Initialize PyMOL
    cmd.reinitialize()

    # Prettify session 
    cmd.do("set orthoscopic, on")
    cmd.do("set ray_trace_fog, 0")
    cmd.do("set depth_cue, 0")
    cmd.do("set antialias, 4")
    cmd.do("set ray_trace_mode, 1")
    cmd.do("set ray_trace_gain, 0.005")
    cmd.do("bg_color white")
    cmd.do("set spec_reflect, 0")
    cmd.do("set transparency, 0.1")  
    cmd.do("set sphere_scale, 2")
    cmd.do("set internal_gui_width, 400")

    PYMOL_OBJECTS = []

    # Load all structures
    for c, (st, p) in enumerate(zip([st1, st2], [p1, p2])):

        uni = st.split("_")[1]

        # Load structure
        cmd.load(os.path.join(PATH_TO_STRUCTURES, uni, f"{st}.pdb"), st)
        cmd.color(COLORS[c], st)
        cmd.show("cartoon", st)

        # Load pocket
        cmd.load(os.path.join(PATH_TO_POCKETS, uni, st, "pockets", f"{p}.pdb"), f"{st}_{p}")
        cmd.color(COLORS[c], f"{st}_{p}")
        cmd.show("spheres", f"{st}_{p}")

        # Copy structure to pocket and remove structure
        cmd.create(f"{st}_{p}", f"{st}_{p} or {st}")
        cmd.delete(st)

        # Append pymol object
        PYMOL_OBJECTS.append(f"{st}_{p}")

    # Align all to reference
    ref = PYMOL_OBJECTS[0]
    for al in PYMOL_OBJECTS[1:]:
        cmd.super(f"{al} and name CA", f"{ref} and name CA")

    # Save PyMOL session
    cmd.reset()
    cmd.save(os.path.join(outdir, f"{pocket1}__{pocket2}.pse"))

In [15]:
pocket1 = "alphafold2_P9WFT1_model_0_pocket_2"
pocket2 = "alphafold3_P9WFU5_model_2_pocket_1"
outdir = "/home/acomajuncosa/Documents/tmp"

prepare_pymol_session(pocket1, pocket2, outdir)

In [ ]:
ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_prioritization/pairs.tsv", sep='\t')
ALL_RESULTS_PAIRS[(ALL_RESULTS_PAIRS['Protein RMSD'] > 10) & (ALL_RESULTS_PAIRS['PocketVec distance'] < 0.17)][:5]

In [ ]:
pocket1 = "alphafold2_P9WFT1_model_0_pocket_2"
pocket2 = "alphafold3_P9WFU5_model_2_pocket_1"
outdir = "/home/acomajuncosa/Documents/tmp"

prepare_pymol_session(pocket1, pocket2, outdir)